In [ ]:
# （空单元格占位：可在此写临时试验代码；当前无执行语句）


# 第 3 周练习 —— 合成数据生成器（双开源模型流式对比）

## 练习目标

用 **Hugging Face** 上的开源 Instruct 模型，做一个能「流式吐出」合成数据的小工具，并用 **Gradio** 并排对比两个模型的输出。

- **编写可生成数据集的提示（prompt）与调用逻辑**
- **同时跑多种模型**（本笔记本默认 LLaMA 3.2 1B 与 Microsoft Phi-4 mini），观察同一 prompt 下风格差异
- **为产品搭 Gradio UI**：输入提示 → 两路 Markdown 流式刷新

## 和本课第 3 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 开源模型本地/Colab 推理 | `AutoModelForCausalLM` + `device_map` |
| 4-bit 量化省显存 | `BitsAndBytesConfig` |
| 流式生成 | `TextIteratorStreamer` + 后台 `Thread` |
| 多模型对比 UI | Gradio `Blocks` 双栏 `Markdown` |

## 怎么跑（Colab）

1. 在 Colab Secrets 配置 `HF_TOKEN`（需有权访问 gated 模型）
2. 从上到下依次运行单元格（先装依赖、再登录、再加载模型）
3. 在 Gradio 文本框粘贴 `user_prompt` / `user_prompt_2` 一类提示，点 Submit 看双模型流式输出
4. 可在 `MODELS` 字典里换成注释区列出的其他 model id 做实验（换模型后需重新加载）


In [ ]:
# ========== 安装/升级推理依赖（Colab shell 魔法） ==========
# bitsandbytes：4-bit 量化；accelerate：device_map 等；transformers 钉版本避免 API 漂移
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6



In [ ]:
# ========== 导入：Colab 密钥、HF、Transformers、Gradio、线程 ==========

# userdata：读 Colab Secrets 里的 HF_TOKEN
from google.colab import userdata
# login：Hugging Face Hub 登录（gated 模型需要）
from huggingface_hub import login
# 分词器/因果 LM/流式解码器/4-bit 量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, TextIteratorStreamer, BitsAndBytesConfig
# PyTorch：张量与 CUDA 检测
import torch
# gc：后面清理显存时用
import gc
# IPython 展示工具（本笔记本主路径用 Gradio，这里先导入备用）
from IPython.display import Markdown, display, update_display
# Gradio：搭双栏对比 UI
import gradio as gr
# Thread：后台跑 model.generate，主线程消费 streamer
from threading import Thread


In [ ]:
# ========== 用 Colab Secret 登录 Hugging Face ==========

# 取出名为 HF_TOKEN 的密钥（Secrets 里的名字必须一致）
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub；add_to_git_credential=True 方便后续拉模型权重
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 模型 id 字典：真正会加载的是 MODELS；其余常量备换 ==========

# 键名 LLAMA / PHI 后面流式循环与 UI 左右栏会用到，勿随意改键
MODELS = {
      "LLAMA":"meta-llama/Llama-3.2-1B-Instruct",
      "PHI":"microsoft/Phi-4-mini-instruct"
}

# 下面是可选替换的 Hugging Face model id（当前未加入 MODELS，不会自动加载）
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MIXTRAL = "mistralai/Mixtral-8x7B-Instruct-v0.1"


In [ ]:
# ========== Prompt A：披萨店评论合成数据（Markdown + 星级） ==========
# 三引号内是发给模型的英文指令，影响输出格式——保持英文原样

user_prompt = """
    You are a synthetic dataset generator. Generate restaurant reviewers for a pizzeria
    cafe. The reviews should be in mark down review the service and the various amenities like valet and many others
    The rating should be returned in a short sentence and a rating out of 5 stars.

    Reply with only the reviews.
"""



In [ ]:
# ========== Prompt B：机器学习概念 Q&A（要求纯 JSON 数组）+ 默认 messages ==========

# 第二个合成任务：严格只要 JSON，不要 markdown 围栏（英文 prompt 勿改）
user_prompt_2 = """
  You are a synthetic dataset generator. Generate exactly 3 high-quality
question-answer pairs about: "Machine Learning".

Requirements:
- Question type: conceptual
- Difficulty: all levels of difficulty low, medium and high
- Each Q&A should be distinct and non-repetitive
- Answers should be accurate, informative, and 1-3 sentences

Respond ONLY with a valid JSON array. No markdown, no explanation. Format:
[
  {"q": "Question text here?", "a": "Answer text here.", "difficulty": "medium"},
]
"""

# 默认对话消息：用 Prompt A（披萨评论）作为 user content；可改成 user_prompt_2 做对比
messages = [
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
# （空单元格占位：可在此试验单次 generate；当前无执行语句）


In [ ]:
# ========== 4-bit 量化配置：同样模型占更少显存 ==========

# BitsAndBytesConfig：告诉 from_pretrained 用 nf4 等 4-bit 方案加载权重
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)


In [ ]:
# ========== 按 MODELS 字典批量加载分词器与量化模型 ==========

# 用名字当键，后面流式函数按 'LLAMA' / 'PHI' 取用
tokenizers = {}
models = {}

# 遍历 MODELS：每个 model_id 各加载一套 tokenizer + CausalLM
for name, model_id in MODELS.items():
    # 下载/缓存分词器
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id)
    # 无 pad_token 时用 eos 顶上，避免 pad 相关警告/错误
    tokenizers[name].pad_token = tokenizers[name].eos_token
    # 加载模型：有 CUDA 用 float16，否则 float32；auto 设备映射 + 上面的 4-bit 配置
    models[name] = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        quantization_config=quant_config
    )

# 加载完成日志
print("Models loaded.")


In [ ]:
# ========== 双模型并行流式生成：yield (llama_text, phi_text) ==========

def call_models_streaming(prompt):
    # Gradio 会反复调用；打印便于在 notebook 日志里追踪
    print("calling models for streaming")
    # 明确使用全局已加载的 models / tokenizers / MODELS
    global models, tokenizers, MODELS

    # 每个模型一个 streamer、一个后台线程
    streamers = {}
    threads = {}
    # 单轮 user 消息：内容就是 UI 传入的 prompt
    messages = [
        {"role": "user", "content": prompt}
    ]

    # 为 MODELS 里每一个模型启动 generate（并行）
    for model_name, model_id in MODELS.items():
        tokenizer = tokenizers[model_name]
        model = models[model_name]

        # 迭代流式解码器：跳过 prompt、跳过特殊 token
        streamer = TextIteratorStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )
        streamers[model_name] = streamer

        # chat template → token ids，并搬到 cuda（本练习假设有 GPU）
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

        # generate 参数：挂上 streamer，采样生成
        generation_kwargs = dict(
            inputs=inputs,
            streamer=streamer,
            max_new_tokens=500,
            do_sample=True,
            temperature=0.7
        )

        # 后台线程跑 generate，主循环下面用 next(streamer) 取 token
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()
        threads[model_name] = thread

    # 两路累计文本与结束标志
    llama_current_text = ""
    phi_current_text = ""
    llama_finished = False
    phi_finished = False

    # 直到两路都 StopIteration
    while not (llama_finished and phi_finished):
        llama_token = None
        phi_token = None

        # 非阻塞式地尝试取下一小段；流结束则标记 finished
        if not llama_finished:
            try:
                llama_token = next(streamers['LLAMA'])
            except StopIteration:
                llama_finished = True

        if not phi_finished:
            try:
                phi_token = next(streamers['PHI'])
            except StopIteration:
                phi_finished = True

        # 有新片段就拼接
        if llama_token:
            llama_current_text += llama_token
        if phi_token:
            phi_current_text += phi_token

        # 每次循环都 yield 当前双栏全文，驱动 Gradio 流式刷新
        yield llama_current_text, phi_current_text

    # 等后台线程真正结束，避免资源悬挂
    for thread in threads.values():
        thread.join()

    print("Streaming completed.")


In [ ]:
# （空单元格占位：可在此写 call_models_streaming 的临时测试；当前无执行语句）



In [ ]:
# ========== Gradio Blocks：一输入框 + 双 Markdown 输出 ==========

# Blocks 上下文：声明式搭界面
with gr.Blocks() as demo:
    # 标题（UI 文案保持英文原样）
    gr.Markdown("# Markdown Comparison UI")

    # 用户输入：合成数据用的 prompt
    prompt = gr.Textbox(
        label="Enter Text",
        placeholder="Type something..."
    )

    # 提交按钮
    submit = gr.Button("Submit")

    # 左右两栏：分别接 LLAMA / PHI 的流式文本
    with gr.Row():
        output1 = gr.Markdown()
        output2 = gr.Markdown()

    # 点击 → 调用 call_models_streaming；outputs 顺序对应 yield 的二元组
    submit.click(
        fn=call_models_streaming,
        inputs=prompt,
        outputs=[output1, output2]
    )

# 启动本地 Gradio 服务（Colab 会给可点链接）
demo.launch()


In [ ]:
# ========== 清理：删掉大对象并清空 CUDA 缓存（家政） ==========

# 尝试删除常见大对象名（若某名字本单元格作用域里不存在会报 NameError——保持原逻辑）
del model, inputs, tokenizer, outputs
# 触发 Python 垃圾回收
gc.collect()
# 释放 PyTorch 缓存的显存块，方便后续再加载别的模型
torch.cuda.empty_cache()
